# Modelo Baseline: Regresión Lineal

Como segundo baseline, antes de entrar en redes neuronales, entrenamos una 
regresión lineal multi-output que sí aprende de toda la ventana de entrada 
(no solo del último día como Buy and Hold).

## Arquitectura

Para cada combinación de ventanas:
1. Se aplana la entrada `X` de forma `(n_muestras, ventana_entrada, 23)` a 
   `(n_muestras, ventana_entrada × 23)`. Por ejemplo, para ventana 30: cada 
   muestra pasa de ser una matriz 30×23 a un vector de 690 valores.
2. Se entrena un modelo `LinearRegression` de scikit-learn que predice 
   directamente los 23 returns promedio de salida.

## Por qué este baseline

- Sí aprende de toda la información de la ventana de entrada, a diferencia 
  de Buy and Hold.
- Es la "frontera mínima" para los modelos neuronales: un MLP es esencialmente 
  una regresión lineal con capas extra y no-linealidades. Si un MLP no supera 
  a la regresión lineal, está mal entrenado o sobredimensionado.

## Imports y carga de datos

In [21]:
import numpy as np
import pandas as pd
import yfinance as yf
import warnings
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

warnings.simplefilter(action="ignore", category=FutureWarning)

# Carga de datos
start_date = '1945-01-01'
tickers_validos = ['AEP', 'BA', 'CAT', 'CNP', 'CVX', 'DIS', 'DTE', 'ED', 'GD',
                   'GE', 'HON', 'HPQ', 'IBM', 'IP', 'JNJ', 'KO', 'KR', 'MMM',
                   'MO', 'MRK', 'MSI', 'PG', 'XOM']

precios_close = yf.download(tickers_validos, start=start_date,
                            auto_adjust=True, progress=False)['Close']
precios_close.dropna(axis=1, inplace=True)

returns = np.log(precios_close / precios_close.shift(1)).dropna()
print(f"Returns: {returns.shape}")

Returns: (16197, 23)


## Función para crear ventanas

In [22]:
def create_time_series_data(data, input_window_size, output_window_size):
    X, y = [], []
    data_array = data.values if isinstance(data, pd.DataFrame) else data
    
    for i in range(len(data_array) - input_window_size - output_window_size + 1):
        input_seq = data_array[i : i + input_window_size]
        output_seq = data_array[i + input_window_size : i + input_window_size + output_window_size]
        X.append(input_seq)
        y.append(np.mean(output_seq, axis=0))
    
    return np.array(X), np.array(y)


def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

## Loop sobre las 16 combinaciones

In [23]:
input_windows = [5, 10, 30, 90]
output_windows = [1, 5, 30, 90]
RANDOM_SEED = 42

resultados = []

for in_w in input_windows:
    for out_w in output_windows:
        # 1. Generar ventanas
        X, y = create_time_series_data(returns, in_w, out_w)
        
       # Split 1: test cronológico (shuffle=False)
        X_train_full, X_test, y_train_full, y_test = train_test_split(
            X, y, test_size=0.1, shuffle=False
        )

        # Split 2: val aleatorio dentro del train (shuffle=True)
        X_train, X_val, y_train, y_val = train_test_split(
            X_train_full, y_train_full, test_size=0.1,
            shuffle=True, random_state=42
        )
        
        # 3. Aplanar: (n, días, activos) -> (n, días*activos)
        X_train_flat = X_train.reshape(X_train.shape[0], -1)
        X_val_flat   = X_val.reshape(X_val.shape[0], -1)
        X_test_flat  = X_test.reshape(X_test.shape[0], -1)
        
        # 4. Entrenar regresión lineal
        modelo = LinearRegression()
        modelo.fit(X_train_flat, y_train)
        
        # 5. Predecir y calcular MAE
        mae_train = mae(y_train, modelo.predict(X_train_flat))
        mae_val   = mae(y_val,   modelo.predict(X_val_flat))
        mae_test  = mae(y_test,  modelo.predict(X_test_flat))
        
        # 6. Número de parámetros: pesos (in_w*23 * 23) + sesgos (23)
        n_params = (in_w * 23) * 23 + 23
        
        resultados.append({
            'input_window': in_w,
            'output_window': out_w,
            'n_params': n_params,
            'mae_train': mae_train,
            'mae_val': mae_val,
            'mae_test': mae_test,
        })

df_resultados = pd.DataFrame(resultados)
df_resultados

,input_window,output_window,n_params,mae_train,mae_val,mae_test
0,5,1,2668,0.011543,0.011624,0.012424
1,5,5,2668,0.005320,0.005289,0.005649
2,5,30,2668,0.002146,0.002131,0.002339
3,5,90,2668,0.001228,0.001242,0.001277
4,10,1,5313,0.011512,0.011798,0.012596
5,10,5,5313,0.005298,0.005311,0.005710
6,10,30,5313,0.002138,0.002120,0.002357
7,10,90,5313,0.001223,0.001248,0.001287
8,30,1,15893,0.011398,0.012185,0.012970
9,30,5,15893,0.005213,0.005523,0.005912


In [24]:
os.makedirs('../results', exist_ok=True)
df_resultados.to_csv('../results/regresion_lineal_resultados.csv', index=False)
print("Guardado en results/regresion_lineal_resultados.csv")

Guardado en results/regresion_lineal_resultados.csv


## Conclusiones

La regresión lineal supera ampliamente a Buy and Hold en todas las 16 
combinaciones, confirmando que aprender de toda la ventana de entrada 
(no solo del último día) aporta señal real. La mejora relativa en MAE test 
va del 34% en la combinación más exigente (5/1) al 90% en las ventanas de 
salida más largas (5/90, 90/90).

Observaciones principales:

- **Más ventana de salida implica menor MAE.** El MAE test pasa de ~0.012 
  (salida 1 día) a ~0.0012 (salida 90 días), un orden de magnitud. Igual 
  que en Buy and Hold, esto refleja la cancelación del ruido al promediar 
  muchos returns: la salida tiende a su media a largo plazo (~0) y se 
  vuelve más fácil de predecir.

- **Más ventana de entrada NO mejora test, e incluso empeora.** Para 
  `output=1` el MAE test pasa de 0.01167 (in=5) a 0.01366 (in=90), un 
  17% peor pese a tener 18 veces más parámetros. La regresión lineal sin 
  regularización empieza a sobreajustar cuando el ratio features/muestras 
  crece. Este efecto es la motivación para usar arquitecturas con 
  regularización implícita (dropout, capas convolucionales con pesos 
  compartidos, etc.) en los siguientes modelos.

- **Aparece un gap train/test (overfitting suave) en input_window=90.** 
  Por ejemplo, en 90/1: train 0.01103 vs test 0.01366, un 24% de diferencia. 
  En las ventanas de entrada cortas (5, 10) los tres conjuntos van casi 
  parejos. Esto da una pista importante para los modelos neuronales: 
  vigilar el control de capacidad cuando la entrada es grande.

- **El número de parámetros crece linealmente con la ventana de entrada.** 
  Pasa de 2.668 (in=5) a 47.633 (in=90). Esto motiva el uso de 
  arquitecturas que comparten pesos a lo largo del tiempo (RNN, CNN), que 
  no escalan así con la longitud de la entrada.

Estos resultados establecen la **frontera mínima exigente** para los modelos 
neuronales: cualquier MLP, RNN, CNN o modelo mixto debería igualar o superar 
estos MAE. Si un modelo neuronal no supera la regresión lineal, está mal 
configurado (típicamente: insuficiente entrenamiento, learning rate 
inadecuado, o sobredimensionado sin regularización).